In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import ndimage
from skimage.morphology import skeletonize
from skimage.graph import pixel_graph
import json
from sklearn.cluster import DBSCAN

W, H, res = 200, 150, 

def make_complex_grid():
    g = np.ones((H, W), dtype=np.uint8) * 255
    def w(y1, y2, x1, x2): g[y1:y2, x1:x2] = 0
    def c(cy, cx, r):
        Y, X = np.ogrid[:H, :W]
        g[(X - cx)**2 + (Y - cy)**2 <= r**2] = 0
    
    w(0, 3, 0, W); w(H-3, H, 0, W); w(0, H, 0, 3); w(0, H, W-3, W)
    w(30, 120, 40, 60); w(30, 120, 140, 160)
    w(10, 30, 10, 40); w(10, 30, 70, 100); w(10, 30, 120, 150)
    w(10, 30, 170, 200); w(10, 30, 220, 250)
    w(50, 70, 80, 120); w(50, 70, 180, 220)
    w(80, 100, 50, 90); w(80, 100, 130, 170); w(80, 100, 210, 250)
    w(120, 140, 70, 110); w(120, 140, 150, 190); w(120, 140, 230, 270)
    c(25, 25, 3); c(25, 175, 3); c(125, 25, 3); c(125, 175, 3)
    c(75, 75, 4); c(175, 75, 4); c(75, 125, 4); c(175, 125, 4)
    w(60, 80, 30, 50); w(60, 80, 190, 210)
    w(100, 120, 30, 50); w(100, 120, 190, 210)
    return g

grid = make_complex_grid()

# ==========================================
# Step 1: 基础骨架与距离图
# ==========================================
free = (grid == 255).astype(np.uint8)
dist = ndimage.distance_transform_edt(free)
skeleton = skeletonize(free).astype(np.uint8)
skel_bool = skeleton.astype(bool)

skel_y, skel_x = np.where(skeleton > 0)
skel_set = set(zip(skel_x.tolist(), skel_y.tolist()))

# ==========================================
# Step 2: 使用 pixel_graph 构建图并提取端点/分叉点
# ==========================================
graph, flat_indices = pixel_graph(
    skel_bool, 
    connectivity=2, 
    edge_function=lambda x, y, d: d
)

degrees = np.diff(graph.indptr)
endpoint_indices = np.where(degrees == 1)[0]
junction_indices = np.where(degrees >= 3)[0]

coord_to_idx = {}
node_xy_coords = []

for i, flat_idx in enumerate(flat_indices):
    y, x = np.unravel_index(flat_idx, skel_bool.shape)
    coord_to_idx[(x, y)] = i
    node_xy_coords.append((y, x))

# ==========================================
# Step 3: 通过向量夹角提取拐弯点
# ==========================================
def get_neighbors(x, y, s_set):
    neis = []
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            if dx == 0 and dy == 0: continue
            if (x+dx, y+dy) in s_set: neis.append((x+dx, y+dy))
    return neis

def walk_straight(sx, sy, prev_x, prev_y, steps, s_set):
    path = []
    cx, cy = sx, sy
    for _ in range(steps):
        neis = get_neighbors(cx, cy, s_set)
        neis = [n for n in neis if n != (prev_x, prev_y)]
        if len(neis) != 1: break 
        path.append(neis[0])
        prev_x, prev_y = cx, cy
        cx, cy = neis[0]
    return path

bend_points_set = set()
ANGLE_THRESHOLD = 45 
MAX_STRAIGHT_STEPS = 15 

for x, y in skel_set:
    neis = get_neighbors(x, y, skel_set)
    if len(neis) != 2: continue 
    
    n1, n2 = neis
    path1 = walk_straight(n1[0], n1[1], x, y, MAX_STRAIGHT_STEPS, skel_set)
    path2 = walk_straight(n2[0], n2[1], x, y, MAX_STRAIGHT_STEPS, skel_set)
    
    if len(path1) > 0 and len(path2) > 0:
        v1 = np.array(path1[-1]) - np.array([x, y])
        v2 = np.array(path2[-1]) - np.array([x, y])
        n1_norm, n2_norm = np.linalg.norm(v1), np.linalg.norm(v2)
        
        if n1_norm > 0 and n2_norm > 0:
            cos_angle = np.dot(v1, v2) / (n1_norm * n2_norm)
            angle = np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0)))
            if angle < (180 - ANGLE_THRESHOLD):
                bend_points_set.add((x, y))

# ==========================================
# Step 4: 合并所有关键节点，并建立 Graph Index
# ==========================================
raw_key_points = set()
for i in endpoint_indices:
    y, x = node_xy_coords[i]
    raw_key_points.add((x, y))
for i in junction_indices:
    y, x = node_xy_coords[i]
    raw_key_points.add((x, y))
raw_key_points.update(bend_points_set)

sampled_key_points = raw_key_points  # 暂时跳过采样

key_point_indices = set()
for x, y in sampled_key_points:
    if (x, y) in coord_to_idx:
        key_point_indices.add(coord_to_idx[(x, y)])

# ==========================================
# Step 5: 构造分叉路径
# ==========================================
def calc_local_span(x, y, s_set, radius=30):
    local_pts = []
    for dx in range(-radius, radius+1):
        for dy in range(-radius, radius+1):
            if (x+dx, y+dy) in s_set: local_pts.append((x+dx, y+dy))
    if not local_pts: return 0.0, 0.0
    pts = np.array(local_pts)
    return round(float((np.max(pts[:, 0]) - np.min(pts[:, 0])) * res), 2), \
           round(float((np.max(pts[:, 1]) - np.min(pts[:, 1])) * res), 2)

def trace_path(start_idx):
    neighbors = graph[start_idx].indices
    connected_nodes = []
    
    for n_idx in neighbors:
        if n_idx in key_point_indices:
            connected_nodes.append(n_idx)
        else:
            prev_idx = start_idx
            curr_idx = n_idx
            max_steps = 200
            steps = 0
            while degrees[curr_idx] == 2 and curr_idx not in key_point_indices and steps < max_steps:
                next_indices = graph[curr_idx].indices
                if len(next_indices) == 1:
                    break
                next_idx = next_indices[0] if next_indices[1] == prev_idx else next_indices[1]
                prev_idx, curr_idx = curr_idx, next_idx
                steps += 1
            
            if curr_idx in key_point_indices:
                connected_nodes.append(curr_idx)
            elif degrees[curr_idx] == 1:
                connected_nodes.append(curr_idx)
                
    return connected_nodes

edges_set = set()
for k_idx in key_point_indices:
    targets = trace_path(k_idx)
    for t_idx in targets:
        edge = tuple(sorted([k_idx, t_idx]))
        edges_set.add(edge)

# ==========================================
# Step 6: 节点合并优化 (修复版)
# ==========================================
def merge_nodes_simple(key_point_indices, node_xy_coords, merge_threshold=5):
    """
    简化版节点合并：直接遍历所有节点对，合并距离小于阈值的节点
    """
    current_indices = set(key_point_indices)
    merged_map = {}
    
    # 转换为 numpy 数组方便计算距离
    points = np.array([node_xy_coords[i] for i in current_indices])
    indices = list(current_indices)
    
    # 遍历所有节点对
    for i in range(len(indices)):
        for j in range(i+1, len(indices)):
            idx1, idx2 = indices[i], indices[j]
            
            # 检查两个节点是否都还在 current_indices 中
            if idx1 not in current_indices or idx2 not in current_indices:
                continue
                
            y1, x1 = node_xy_coords[idx1]
            y2, x2 = node_xy_coords[idx2]
            distance = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
            
            if distance < merge_threshold:
                # 将 idx2 合并到 idx1
                merged_map[idx2] = idx1
                current_indices.remove(idx2)
    
    # 重新构建边集合
    final_edges = set()
    for edge in edges_set:
        # 替换被合并的节点
        if edge[0] in merged_map:
            edge = (merged_map[edge[0]], edge[1])
        if edge[1] in merged_map:
            edge = (edge[0], merged_map[edge[1]])
        
        # 确保边的两个端点都在当前节点集合中
        if edge[0] in current_indices and edge[1] in current_indices:
            final_edges.add(edge)
    
    return current_indices, final_edges

# 执行节点合并
merge_threshold = 5
# key_point_indices, edges_set = merge_nodes_simple(key_point_indices, node_xy_coords, merge_threshold)

# ==========================================
# 生成最终节点和边的 JSON 结构
# ==========================================
nodes = []
node_id_map_idx = {}
idx = 0
for k_idx in key_point_indices:
    y, x = node_xy_coords[k_idx]
    sx, sy = calc_local_span(x, y, skel_set)
    node_id = f"N_{idx}"
    nodes.append({
        "id": node_id,
        "x": round(float(x * res), 2),
        "y": round(float(y * res), 2),
        "span_x": sx,
        "span_y": sy
    })
    node_id_map_idx[k_idx] = node_id
    idx += 1

edges_json = []
for edge in edges_set:
    if edge[0] in node_id_map_idx and edge[1] in node_id_map_idx:
        edges_json.append({
            "from": node_id_map_idx[edge[0]],
            "to": node_id_map_idx[edge[1]]
        })

print(f"合并后提取到 {len(nodes)} 个纯导航节点 (含拐点)")
print(f"合并后提取到 {len(edges_json)} 条分叉/直线路径边")

# ==========================================
# 可视化
# ==========================================
node_dict = {n["id"]: n for n in nodes}

fig, ax = plt.subplots(figsize=(15, 10))
ax.imshow(grid, cmap='gray', origin='upper', alpha=0.5)
ax.scatter(skel_x, skel_y, s=0.5, color='lightgray', alpha=0.6)

for n in nodes:
    px, py = n['x'] / res, n['y'] / res
    width_px = max(4, min(n['span_x'] * 10, 25))  
    height_px = max(4, min(n['span_y'] * 10, 25)) 
    
    ax.plot(px, py, 's', markersize=width_px, markeredgewidth=1.5, 
            color='red', markeredgecolor='black')
    ax.annotate(f"{n['span_x']}x{n['span_y']}", (px, py),
                textcoords="offset points", xytext=(8, 8), fontsize=7, color='darkred')

for e in edges_json:
    n1, n2 = node_dict[e["from"]], node_dict[e["to"]]
    ax.plot([n1['x']/res, n2['x']/res], [n1['y']/res, n2['y']/res], 'g-', linewidth=2, alpha=0.8)

ax.set_title("Pixel Graph Based Topology\n(Red=Key Nodes, Green=Traced Fork Paths)", fontsize=13)
ax.set_xlim(0, W); ax.set_ylim(H, 0)
plt.tight_layout()
plt.savefig('pixel_graph_topology.png', dpi=150, bbox_inches='tight')
plt.close()

# ==========================================
# 输出 LLM JSON
# ==========================================
print("\n" + "=" * 50)
print("JSON for LLM:")
print("=" * 50)
print(json.dumps({"nodes": nodes, "edges": edges_json}, indent=2, ensure_ascii=False))


合并后提取到 184 个纯导航节点 (含拐点)
合并后提取到 218 条分叉/直线路径边

JSON for LLM:
{
  "nodes": [
    {
      "id": "N_0",
      "x": 66.0,
      "y": 64.0,
      "span_x": 32.0,
      "span_y": 45.0
    },
    {
      "id": "N_1",
      "x": 193.0,
      "y": 139.0,
      "span_x": 30.0,
      "span_y": 34.0
    },
    {
      "id": "N_2",
      "x": 8.0,
      "y": 6.0,
      "span_x": 32.0,
      "span_y": 30.0
    },
    {
      "id": "N_3",
      "x": 132.0,
      "y": 64.0,
      "span_x": 32.0,
      "span_y": 60.0
    },
    {
      "id": "N_4",
      "x": 10.0,
      "y": 6.0,
      "span_x": 34.0,
      "span_y": 30.0
    },
    {
      "id": "N_5",
      "x": 193.0,
      "y": 140.0,
      "span_x": 30.0,
      "span_y": 33.0
    },
    {
      "id": "N_6",
      "x": 12.0,
      "y": 6.0,
      "span_x": 36.0,
      "span_y": 30.0
    },
    {
      "id": "N_7",
      "x": 11.0,
      "y": 6.0,
      "span_x": 35.0,
      "span_y": 30.0
    },
    {
      "id": "N_8",
      "x": 14.0,
      "y": 

In [39]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import ndimage
from skimage.morphology import skeletonize
from skimage.graph import pixel_graph
import json
from sklearn.cluster import DBSCAN

W, H, res = 200, 150, 0.1

def make_complex_grid():
    g = np.ones((H, W), dtype=np.uint8) * 255
    def w(y1, y2, x1, x2): g[y1:y2, x1:x2] = 0
    def c(cy, cx, r):
        Y, X = np.ogrid[:H, :W]
        g[(X - cx)**2 + (Y - cy)**2 <= r**2] = 0
    
    w(0, 3, 0, W); w(H-3, H, 0, W); w(0, H, 0, 3); w(0, H, W-3, W)
    w(30, 120, 40, 60); w(30, 120, 140, 160)
    w(10, 30, 10, 40); w(10, 30, 70, 100); w(10, 30, 120, 150)
    w(10, 30, 170, 200); w(10, 30, 220, 250)
    w(50, 70, 80, 120); w(50, 70, 180, 220)
    w(80, 100, 50, 90); w(80, 100, 130, 170); w(80, 100, 210, 250)
    w(120, 140, 70, 110); w(120, 140, 150, 190); w(120, 140, 230, 270)
    c(25, 25, 3); c(25, 175, 3); c(125, 25, 3); c(125, 175, 3)
    c(75, 75, 4); c(175, 75, 4); c(75, 125, 4); c(175, 125, 4)
    w(60, 80, 30, 50); w(60, 80, 190, 210)
    w(100, 120, 30, 50); w(100, 120, 190, 210)
    return g

grid = make_complex_grid()

# ==========================================
# Step 1: 基础骨架与距离图
# ==========================================
free = (grid == 255).astype(np.uint8)
dist = ndimage.distance_transform_edt(free)
skeleton = skeletonize(free).astype(np.uint8)
skel_bool = skeleton.astype(bool)

skel_y, skel_x = np.where(skeleton > 0)
skel_set = set(zip(skel_x.tolist(), skel_y.tolist()))

# ==========================================
# Step 2: 使用 pixel_graph 构建图并提取端点/分叉点
# ==========================================
graph, flat_indices = pixel_graph(
    skel_bool, 
    connectivity=2, 
    edge_function=lambda x, y, d: d
)

degrees = np.diff(graph.indptr)
endpoint_indices = np.where(degrees == 1)[0]
junction_indices = np.where(degrees >= 3)[0]

coord_to_idx = {}
node_xy_coords = []

for i, flat_idx in enumerate(flat_indices):
    y, x = np.unravel_index(flat_idx, skel_bool.shape)
    coord_to_idx[(x, y)] = i
    node_xy_coords.append((y, x))

# ==========================================
# Step 3: 通过向量夹角提取拐弯点
# ==========================================
def get_neighbors(x, y, s_set):
    neis = []
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            if dx == 0 and dy == 0: continue
            if (x+dx, y+dy) in s_set: neis.append((x+dx, y+dy))
    return neis

def walk_straight(sx, sy, prev_x, prev_y, steps, s_set):
    path = []
    cx, cy = sx, sy
    for _ in range(steps):
        neis = get_neighbors(cx, cy, s_set)
        neis = [n for n in neis if n != (prev_x, prev_y)]
        if len(neis) != 1: break 
        path.append(neis[0])
        prev_x, prev_y = cx, cy
        cx, cy = neis[0]
    return path

bend_points_set = set()
ANGLE_THRESHOLD = 45 
MAX_STRAIGHT_STEPS = 15 

for x, y in skel_set:
    neis = get_neighbors(x, y, skel_set)
    if len(neis) != 2: continue 
    
    n1, n2 = neis
    path1 = walk_straight(n1[0], n1[1], x, y, MAX_STRAIGHT_STEPS, skel_set)
    path2 = walk_straight(n2[0], n2[1], x, y, MAX_STRAIGHT_STEPS, skel_set)
    
    if len(path1) > 0 and len(path2) > 0:
        v1 = np.array(path1[-1]) - np.array([x, y])
        v2 = np.array(path2[-1]) - np.array([x, y])
        n1_norm, n2_norm = np.linalg.norm(v1), np.linalg.norm(v2)
        
        if n1_norm > 0 and n2_norm > 0:
            cos_angle = np.dot(v1, v2) / (n1_norm * n2_norm)
            angle = np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0)))
            if angle < (180 - ANGLE_THRESHOLD):
                bend_points_set.add((x, y))

# ==========================================
# Step 4: 合并所有关键节点，并建立 Graph Index
# ==========================================
raw_key_points = set()
for i in endpoint_indices:
    y, x = node_xy_coords[i]
    raw_key_points.add((x, y))
for i in junction_indices:
    y, x = node_xy_coords[i]
    raw_key_points.add((x, y))
raw_key_points.update(bend_points_set)

sampled_key_points = raw_key_points  # 暂时跳过采样

key_point_indices = set()
for x, y in sampled_key_points:
    if (x, y) in coord_to_idx:
        key_point_indices.add(coord_to_idx[(x, y)])

# ==========================================
# Step 5: 构造分叉路径
# ==========================================
def calc_local_span(x, y, s_set, radius=30):
    local_pts = []
    for dx in range(-radius, radius+1):
        for dy in range(-radius, radius+1):
            if (x+dx, y+dy) in s_set: local_pts.append((x+dx, y+dy))
    if not local_pts: return 0.0, 0.0
    pts = np.array(local_pts)
    return round(float((np.max(pts[:, 0]) - np.min(pts[:, 0])) * res), 2), \
           round(float((np.max(pts[:, 1]) - np.min(pts[:, 1])) * res), 2)

def trace_path(start_idx):
    neighbors = graph[start_idx].indices
    connected_nodes = []
    
    for n_idx in neighbors:
        if n_idx in key_point_indices:
            connected_nodes.append(n_idx)
        else:
            prev_idx = start_idx
            curr_idx = n_idx
            max_steps = 200
            steps = 0
            while degrees[curr_idx] == 2 and curr_idx not in key_point_indices and steps < max_steps:
                next_indices = graph[curr_idx].indices
                if len(next_indices) == 1:
                    break
                next_idx = next_indices[0] if next_indices[1] == prev_idx else next_indices[1]
                prev_idx, curr_idx = curr_idx, next_idx
                steps += 1
            
            if curr_idx in key_point_indices:
                connected_nodes.append(curr_idx)
            elif degrees[curr_idx] == 1:
                connected_nodes.append(curr_idx)
                
    return connected_nodes

edges_set = set()
for k_idx in key_point_indices:
    targets = trace_path(k_idx)
    for t_idx in targets:
        edge = tuple(sorted([k_idx, t_idx]))
        edges_set.add(edge)

# ==========================================
# Step 6: 节点合并优化 (修复版)
# ==========================================
def merge_nodes_simple(key_point_indices, node_xy_coords, merge_threshold=5):
    """
    简化版节点合并：直接遍历所有节点对，合并距离小于阈值的节点
    """
    current_indices = set(key_point_indices)
    merged_map = {}
    
    # 转换为 numpy 数组方便计算距离
    points = np.array([node_xy_coords[i] for i in current_indices])
    indices = list(current_indices)
    
    # 遍历所有节点对
    for i in range(len(indices)):
        for j in range(i+1, len(indices)):
            idx1, idx2 = indices[i], indices[j]
            
            # 检查两个节点是否都还在 current_indices 中
            if idx1 not in current_indices or idx2 not in current_indices:
                continue
                
            y1, x1 = node_xy_coords[idx1]
            y2, x2 = node_xy_coords[idx2]
            distance = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
            
            if distance < merge_threshold:
                # 将 idx2 合并到 idx1
                merged_map[idx2] = idx1
                current_indices.remove(idx2)
    
    # 重新构建边集合
    final_edges = set()
    for edge in edges_set:
        # 替换被合并的节点
        if edge[0] in merged_map:
            edge = (merged_map[edge[0]], edge[1])
        if edge[1] in merged_map:
            edge = (edge[0], merged_map[edge[1]])
        
        # 确保边的两个端点都在当前节点集合中
        if edge[0] in current_indices and edge[1] in current_indices:
            final_edges.add(edge)
    
    return current_indices, final_edges

# 执行节点合并
merge_threshold = 5
key_point_indices, edges_set = merge_nodes_simple(key_point_indices, node_xy_coords, merge_threshold)

# ==========================================
# 生成最终节点和边的 JSON 结构
# ==========================================
nodes = []
node_id_map_idx = {}
idx = 0
for k_idx in key_point_indices:
    y, x = node_xy_coords[k_idx]
    sx, sy = calc_local_span(x, y, skel_set)
    node_id = f"N_{idx}"
    nodes.append({
        "id": node_id,
        "x": round(float(x * res), 2),
        "y": round(float(y * res), 2),
        "span_x": sx,
        "span_y": sy
    })
    node_id_map_idx[k_idx] = node_id
    idx += 1

edges_json = []
for edge in edges_set:
    if edge[0] in node_id_map_idx and edge[1] in node_id_map_idx:
        edges_json.append({
            "from": node_id_map_idx[edge[0]],
            "to": node_id_map_idx[edge[1]]
        })

print(f"合并后提取到 {len(nodes)} 个纯导航节点 (含拐点)")
print(f"合并后提取到 {len(edges_json)} 条分叉/直线路径边")

# ==========================================
# 可视化
# ==========================================
node_dict = {n["id"]: n for n in nodes}

fig, ax = plt.subplots(figsize=(15, 10))
ax.imshow(grid, cmap='gray', origin='upper', alpha=0.5)
ax.scatter(skel_x, skel_y, s=0.5, color='lightgray', alpha=0.6)

for n in nodes:
    px, py = n['x'] / res, n['y'] / res
    # width_px = max(4, min(n['span_x'] * 10, 25))  
    # height_px = max(4, min(n['span_y'] * 10, 25)) 
    width_px = n['span_x'] * 10 
    height_px = n['span_y'] * 10
    
    ax.plot(px, py, 's', markersize=width_px, markeredgewidth=1.5, 
            color='red', markeredgecolor='black')
    ax.annotate(f"{n['span_x']}x{n['span_y']}", (px, py),
                textcoords="offset points", xytext=(8, 8), fontsize=7, color='darkred')

for e in edges_json:
    n1, n2 = node_dict[e["from"]], node_dict[e["to"]]
    ax.plot([n1['x']/res, n2['x']/res], [n1['y']/res, n2['y']/res], 'g-', linewidth=2, alpha=0.8)

ax.set_title("Pixel Graph Based Topology\n(Red=Key Nodes, Green=Traced Fork Paths)", fontsize=13)
ax.set_xlim(0, W); ax.set_ylim(H, 0)
plt.tight_layout()
plt.savefig('pixel_graph_topology_merged.png', dpi=150, bbox_inches='tight')
plt.close()

# ==========================================
# 输出 LLM JSON
# ==========================================
print("\n" + "=" * 50)
print("JSON for LLM:")
print("=" * 50)
print(json.dumps({"nodes": nodes, "edges": edges_json}, indent=2, ensure_ascii=False))


合并后提取到 47 个纯导航节点 (含拐点)
合并后提取到 91 条分叉/直线路径边

JSON for LLM:
{
  "nodes": [
    {
      "id": "N_0",
      "x": 6.6,
      "y": 6.4,
      "span_x": 3.2,
      "span_y": 4.5
    },
    {
      "id": "N_1",
      "x": 19.3,
      "y": 13.9,
      "span_x": 3.0,
      "span_y": 3.4
    },
    {
      "id": "N_2",
      "x": 0.8,
      "y": 0.6,
      "span_x": 3.2,
      "span_y": 3.0
    },
    {
      "id": "N_3",
      "x": 13.2,
      "y": 6.4,
      "span_x": 3.2,
      "span_y": 6.0
    },
    {
      "id": "N_4",
      "x": 1.4,
      "y": 0.6,
      "span_x": 3.8,
      "span_y": 3.0
    },
    {
      "id": "N_5",
      "x": 6.5,
      "y": 6.9,
      "span_x": 3.0,
      "span_y": 4.0
    },
    {
      "id": "N_6",
      "x": 19.4,
      "y": 0.5,
      "span_x": 3.0,
      "span_y": 3.0
    },
    {
      "id": "N_7",
      "x": 13.4,
      "y": 6.9,
      "span_x": 3.0,
      "span_y": 6.0
    },
    {
      "id": "N_8",
      "x": 6.6,
      "y": 7.4,
      "span_x": 3.1,
    

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import ndimage
from skimage.morphology import skeletonize
from skimage.graph import pixel_graph
import json
from sklearn.cluster import DBSCAN

# W, H, res = 200, 150, 0.25

# def make_complex_grid():
#     g = np.ones((H, W), dtype=np.uint8) * 255
#     def w(y1, y2, x1, x2): g[y1:y2, x1:x2] = 0
#     def c(cy, cx, r):
#         Y, X = np.ogrid[:H, :W]
#         g[(X - cx)**2 + (Y - cy)**2 <= r**2] = 0
    
#     w(0, 3, 0, W); w(H-3, H, 0, W); w(0, H, 0, 3); w(0, H, W-3, W)
#     w(30, 120, 40, 60); w(30, 120, 140, 160)
#     w(10, 30, 10, 40); w(10, 30, 70, 100); w(10, 30, 120, 150)
#     w(10, 30, 170, 200); w(10, 30, 220, 250)
#     w(50, 70, 80, 120); w(50, 70, 180, 220)
#     w(80, 100, 50, 90); w(80, 100, 130, 170); w(80, 100, 210, 250)
#     w(120, 140, 70, 110); w(120, 140, 150, 190); w(120, 140, 230, 270)
#     c(25, 25, 3); c(25, 175, 3); c(125, 25, 3); c(125, 175, 3)
#     c(75, 75, 4); c(175, 75, 4); c(75, 125, 4); c(175, 125, 4)
#     w(60, 80, 30, 50); w(60, 80, 190, 210)
#     w(100, 120, 30, 50); w(100, 120, 190, 210)
#     return g

# grid = make_complex_grid()

W, H, res = 250, 200, 0.05  # ★ 放大画布、提高分辨率

def make_complex_grid_v2():
    g = np.ones((H, W), dtype=np.uint8) * 255
    def w(y1, y2, x1, x2): g[y1:y2, x1:x2] = 0
    def c(cy, cx, r):
        Y, X = np.ogrid[:H, :W]
        g[(X - cx)**2 + (Y - cy)**2 <= r**2] = 0

    # ============================================================
    # 1. 外墙边界
    # ============================================================
    w(0, 3, 0, W); w(H-3, H, 0, W)
    w(0, H, 0, 3); w(0, H, W-3, W)

    # ============================================================
    # 2. 主干道 —— 宽窄混合
    # ============================================================
    # 主横廊（宽）
    w(40, 65, 15, 235)
    # 主纵廊（宽）
    w(15, 185, 50, 75)
    # 次横廊（窄）
    w(110, 125, 15, 235)
    # 次纵廊（窄）
    w(65, 135, 120, 135)

    # ============================================================
    # 3. 对角通道（45° 方向）
    # ============================================================
    for i in range(50):
        # 左上 -> 右下 对角 1
        d = 3
        x = 80 + i
        y = 70 + i
        if 0 <= y < H and 0 <= x < W:
            w(y, y+d, x, x+d)
        # 右上 -> 左下 对角 2
        x2 = 200 - i
        y2 = 70 + i
        if 0 <= y2 < H and 0 <= x2 < W:
            w(y2, y2+d, x2-3, x2)
        # 短对角 次级
        x3 = 85 + i
        y3 = 140 + i
        if 0 <= y3 < H and 0 <= x3 < W:
            w(y3, y3+2, x3, x3+2)

    # ============================================================
    # 4. 不规则形状房间
    # ============================================================
    # L 形房间（左下）
    w(145, 175, 15, 45)
    w(160, 175, 15, 80)

    # T 形房间（右上）
    w(15, 40, 150, 180)
    w(15, 40, 175, 230)

    # Z 形走廊
    w(80, 95, 15, 45)
    w(95, 110, 30, 55)
    w(110, 125, 15, 40)

    # U 形房间（右下）
    w(145, 185, 155, 175)
    w(145, 185, 210, 230)
    w(165, 185, 155, 230)

    # 十字形房间（中下）
    w(155, 175, 85, 135)
    w(145, 185, 100, 120)

    # ============================================================
    # 5. 弧形弯道（用密集圆孔模拟弧线通道）
    # ============================================================
    center_y, center_x = 130, 230
    for angle_deg in range(200, 340):
        angle = np.radians(angle_deg)
        r = 18
        cx = int(center_x + r * np.cos(angle))
        cy = int(center_y + r * np.sin(angle))
        c(cy, cx, 4)

    # 第二段弧形（更大半径）
    center_y2, center_x2 = 175, 100
    for angle_deg in range(-90, 30):
        angle = np.radians(angle_deg)
        r = 22
        cx = int(center_x2 + r * np.cos(angle))
        cy = int(center_y2 + r * np.sin(angle))
        c(cy, cx, 3)

    # ============================================================
    # 6. 圆形障碍物 + 柱子
    # ============================================================
    # 大圆洞
    c(50, 100, 8)
    c(50, 160, 8)
    c(95, 165, 6)
    # 小柱子群
    for cy, cx in [(52, 130), (52, 140), (60, 130), (60, 140),
                    (120, 70), (120, 80), (130, 70), (130, 80),
                    (85, 195), (95, 195)]:
        c(cy, cx, 2)

    # ============================================================
    # 7. 菱形通道（通过旋转走廊近似）
    # ============================================================
    for i in range(35):
        t = i / 35.0 * np.pi * 0.5
        cx = int(160 + 25 * np.cos(t + np.pi/4))
        cy = int(85 + 25 * np.sin(t + np.pi/4))
        c(cy, cx, 3)

    # ============================================================
    # 8. 锥形走廊（逐渐变窄）
    # ============================================================
    for i in range(30):
        y = 25 + i
        half_w = max(2, 10 - i // 3)
        cx = 225
        w(y, y+1, cx - half_w, cx + half_w)

    # ============================================================
    # 9. 末端死胡同（各种朝向）
    # ============================================================
    # 朝上的
    w(140, 160, 40, 48)
    # 朝下的
    w(70, 80, 185, 193)
    # 朝左的
    w(42, 48, 60, 80)
    # 朝右的
    w(55, 61, 135, 150)

    # ============================================================
    # 10. 多通交叉路口
    # ============================================================
    # 五岔口
    c(125, 50, 5)
    w(125, 130, 35, 50)
    w(125, 130, 50, 65)

    return g


# ============================================================
# 运行完整流程（与之前完全一致）
# ============================================================
grid = make_complex_grid_v2()

# ==========================================
# Step 1: 基础骨架与距离图
# ==========================================
free = (grid == 255).astype(np.uint8)
dist = ndimage.distance_transform_edt(free)
skeleton = skeletonize(free).astype(np.uint8)
skel_bool = skeleton.astype(bool)

skel_y, skel_x = np.where(skeleton > 0)
skel_set = set(zip(skel_x.tolist(), skel_y.tolist()))

# ==========================================
# Step 2: 使用 pixel_graph 构建图并提取端点/分叉点
# ==========================================
graph, flat_indices = pixel_graph(
    skel_bool, 
    connectivity=2, 
    edge_function=lambda x, y, d: d
)

degrees = np.diff(graph.indptr)
endpoint_indices = np.where(degrees == 1)[0]
junction_indices = np.where(degrees >= 3)[0]

coord_to_idx = {}
node_xy_coords = []

for i, flat_idx in enumerate(flat_indices):
    y, x = np.unravel_index(flat_idx, skel_bool.shape)
    coord_to_idx[(x, y)] = i
    node_xy_coords.append((y, x))

# ==========================================
# Step 3: 通过向量夹角提取拐弯点
# ==========================================
def get_neighbors(x, y, s_set):
    neis = []
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            if dx == 0 and dy == 0: continue
            if (x+dx, y+dy) in s_set: neis.append((x+dx, y+dy))
    return neis

def walk_straight(sx, sy, prev_x, prev_y, steps, s_set):
    path = []
    cx, cy = sx, sy
    for _ in range(steps):
        neis = get_neighbors(cx, cy, s_set)
        neis = [n for n in neis if n != (prev_x, prev_y)]
        if len(neis) != 1: break 
        path.append(neis[0])
        prev_x, prev_y = cx, cy
        cx, cy = neis[0]
    return path

bend_points_set = set()
ANGLE_THRESHOLD = 20
MAX_STRAIGHT_STEPS = 10

for x, y in skel_set:
    neis = get_neighbors(x, y, skel_set)
    if len(neis) != 2: continue 
    
    n1, n2 = neis
    path1 = walk_straight(n1[0], n1[1], x, y, MAX_STRAIGHT_STEPS, skel_set)
    path2 = walk_straight(n2[0], n2[1], x, y, MAX_STRAIGHT_STEPS, skel_set)
    
    if len(path1) > 0 and len(path2) > 0:
        v1 = np.array(path1[-1]) - np.array([x, y])
        v2 = np.array(path2[-1]) - np.array([x, y])
        n1_norm, n2_norm = np.linalg.norm(v1), np.linalg.norm(v2)
        
        if n1_norm > 0 and n2_norm > 0:
            cos_angle = np.dot(v1, v2) / (n1_norm * n2_norm)
            angle = np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0)))
            if angle < (180 - ANGLE_THRESHOLD):
                bend_points_set.add((x, y))

# ==========================================
# Step 4: 合并所有关键节点，并建立 Graph Index
# ==========================================
raw_key_points = set()
for i in endpoint_indices:
    y, x = node_xy_coords[i]
    raw_key_points.add((x, y))
for i in junction_indices:
    y, x = node_xy_coords[i]
    raw_key_points.add((x, y))
raw_key_points.update(bend_points_set)

sampled_key_points = raw_key_points  # 暂时跳过采样

key_point_indices = set()
for x, y in sampled_key_points:
    if (x, y) in coord_to_idx:
        key_point_indices.add(coord_to_idx[(x, y)])

# ==========================================
# Step 5: 构造分叉路径
# ==========================================
def calc_local_span(x, y, s_set, radius=30):
    local_pts = []
    for dx in range(-radius, radius+1):
        for dy in range(-radius, radius+1):
            if (x+dx, y+dy) in s_set: local_pts.append((x+dx, y+dy))
    if not local_pts: return 0.0, 0.0
    pts = np.array(local_pts)
    return round(float((np.max(pts[:, 0]) - np.min(pts[:, 0])) * res), 2), \
           round(float((np.max(pts[:, 1]) - np.min(pts[:, 1])) * res), 2)

def trace_path(start_idx):
    neighbors = graph[start_idx].indices
    connected_nodes = []
    
    for n_idx in neighbors:
        if n_idx in key_point_indices:
            connected_nodes.append(n_idx)
        else:
            prev_idx = start_idx
            curr_idx = n_idx
            max_steps = 200
            steps = 0
            while degrees[curr_idx] == 2 and curr_idx not in key_point_indices and steps < max_steps:
                next_indices = graph[curr_idx].indices
                if len(next_indices) == 1:
                    break
                next_idx = next_indices[0] if next_indices[1] == prev_idx else next_indices[1]
                prev_idx, curr_idx = curr_idx, next_idx
                steps += 1
            
            if curr_idx in key_point_indices:
                connected_nodes.append(curr_idx)
            elif degrees[curr_idx] == 1:
                connected_nodes.append(curr_idx)
                
    return connected_nodes

edges_set = set()
for k_idx in key_point_indices:
    targets = trace_path(k_idx)
    for t_idx in targets:
        edge = tuple(sorted([k_idx, t_idx]))
        edges_set.add(edge)

# ==========================================
# Step 6: 节点合并优化 (修复版)
# ==========================================
def merge_nodes_simple(key_point_indices, node_xy_coords, merge_threshold=5):
    """
    简化版节点合并：直接遍历所有节点对，合并距离小于阈值的节点
    """
    current_indices = set(key_point_indices)
    merged_map = {}
    
    # 转换为 numpy 数组方便计算距离
    points = np.array([node_xy_coords[i] for i in current_indices])
    indices = list(current_indices)
    
    # 遍历所有节点对
    for i in range(len(indices)):
        for j in range(i+1, len(indices)):
            idx1, idx2 = indices[i], indices[j]
            
            # 检查两个节点是否都还在 current_indices 中
            if idx1 not in current_indices or idx2 not in current_indices:
                continue
                
            y1, x1 = node_xy_coords[idx1]
            y2, x2 = node_xy_coords[idx2]
            distance = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
            
            if distance < merge_threshold:
                # 将 idx2 合并到 idx1
                merged_map[idx2] = idx1
                current_indices.remove(idx2)
    
    # 重新构建边集合
    final_edges = set()
    for edge in edges_set:
        # 替换被合并的节点
        if edge[0] in merged_map:
            edge = (merged_map[edge[0]], edge[1])
        if edge[1] in merged_map:
            edge = (edge[0], merged_map[edge[1]])
        
        # 确保边的两个端点都在当前节点集合中
        if edge[0] in current_indices and edge[1] in current_indices:
            final_edges.add(edge)
    
    return current_indices, final_edges

# 执行节点合并
merge_threshold = 5
key_point_indices, edges_set = merge_nodes_simple(key_point_indices, node_xy_coords, merge_threshold)

# ==========================================
# 生成最终节点和边的 JSON 结构
# ==========================================
nodes = []
node_id_map_idx = {}
idx = 0
for k_idx in key_point_indices:
    y, x = node_xy_coords[k_idx]
    sx, sy = calc_local_span(x, y, skel_set)
    node_id = f"N_{idx}"
    nodes.append({
        "id": node_id,
        "x": round(float(x * res), 2),
        "y": round(float(y * res), 2),
        "span_x": sx,
        "span_y": sy
    })
    node_id_map_idx[k_idx] = node_id
    idx += 1

edges_json = []
for edge in edges_set:
    if edge[0] in node_id_map_idx and edge[1] in node_id_map_idx:
        edges_json.append({
            "from": node_id_map_idx[edge[0]],
            "to": node_id_map_idx[edge[1]]
        })

print(f"合并后提取到 {len(nodes)} 个纯导航节点 (含拐点)")
print(f"合并后提取到 {len(edges_json)} 条分叉/直线路径边")

# ==========================================
# 可视化
# ==========================================
node_dict = {n["id"]: n for n in nodes}

fig, ax = plt.subplots(figsize=(15, 10))
ax.imshow(grid, cmap='gray', origin='upper', alpha=0.5)
ax.scatter(skel_x, skel_y, s=0.5, color='lightgray', alpha=0.6)

import matplotlib.patches as patches

# ... 前面的代码不变 ...

for n in nodes:
    px, py = n['x'] / res, n['y'] / res
    
    # 注意：这里假设你想在 数据坐标系 下体现真实的长宽比例
    # 如果你依然想保持在屏幕上固定视觉大小(点坐标)，需要做坐标转换
    # 这里提供两种思路：

    # ================= 思路一：在数据坐标系中画真实比例（推荐用于拓扑图） =================
    # 直接将 span_x, span_y 作为数据坐标的宽高
    w_data = n['span_x']
    h_data = n['span_y']
    
    # 创建矩形，注意 (px, py) 是中心点，Rectangle 的 xy 参数是左下角
    rect = patches.Rectangle(
        (px - w_data/2, py - h_data/2),  # 左下角坐标
        w_data,                          # 宽度
        h_data,                          # 高度
        linewidth=1.5, 
        edgecolor='black', 
        facecolor='red', 
        alpha=0.8
    )
    ax.add_patch(rect)
    
    # 标注
    ax.annotate(f"{n['span_x']:.1f}x{n['span_y']:.1f}", (px, py),
                textcoords="offset points", xytext=(8, 8), fontsize=7, color='darkred')
    # 只添加节点ID指标
    ax.annotate(n['id'], (px, py),
                textcoords="offset points", xytext=(8, 8), fontsize=7, color='darkred')

    # ================= 思路二：在屏幕显示坐标系中保持你原来的大小限制逻辑 =================
    # 如果你的 span 是归一化的极小值（如0.05），直接画在数据坐标里会看不见，
    # 你必须将其转换为 Display 坐标（点 points），但这在 matplotlib 中非常繁琐，
    # 通常需要借助 inset_axes 或复杂的 transform。

for e in edges_json:
    n1, n2 = node_dict[e["from"]], node_dict[e["to"]]
    ax.plot([n1['x']/res, n2['x']/res], [n1['y']/res, n2['y']/res], 'g-', linewidth=2, alpha=0.8)

ax.set_title("Pixel Graph Based Topology\n(Red=Key Nodes, Green=Traced Fork Paths)", fontsize=13)
ax.set_xlim(0, W); ax.set_ylim(H, 0)
plt.tight_layout()
plt.savefig('pixel_graph_topology_merged.png', dpi=150, bbox_inches='tight')
plt.close()

# ==========================================
# 输出 LLM JSON
# ==========================================
print("\n" + "=" * 50)
print("JSON for LLM:")
print("=" * 50)
print(json.dumps({"nodes": nodes, "edges": edges_json}, indent=2, ensure_ascii=False))


合并后提取到 176 个纯导航节点 (含拐点)
合并后提取到 333 条分叉/直线路径边

JSON for LLM:
{
  "nodes": [
    {
      "id": "N_0",
      "x": 2.2,
      "y": 0.4,
      "span_x": 3.0,
      "span_y": 0.95
    },
    {
      "id": "N_1",
      "x": 2.45,
      "y": 0.4,
      "span_x": 3.0,
      "span_y": 0.7
    },
    {
      "id": "N_2",
      "x": 3.7,
      "y": 0.4,
      "span_x": 3.0,
      "span_y": 0.65
    },
    {
      "id": "N_3",
      "x": 3.95,
      "y": 0.4,
      "span_x": 3.0,
      "span_y": 0.65
    },
    {
      "id": "N_4",
      "x": 7.2,
      "y": 0.4,
      "span_x": 3.0,
      "span_y": 0.65
    },
    {
      "id": "N_5",
      "x": 7.45,
      "y": 0.4,
      "span_x": 3.0,
      "span_y": 0.65
    },
    {
      "id": "N_6",
      "x": 11.4,
      "y": 0.4,
      "span_x": 2.1,
      "span_y": 1.5
    },
    {
      "id": "N_7",
      "x": 11.65,
      "y": 0.4,
      "span_x": 1.85,
      "span_y": 1.5
    },
    {
      "id": "N_8",
      "x": 4.15,
      "y": 0.55,
      "span_x"

In [ ]:
map_diagram = {"nodes": nodes, "edges": edges_json}

In [2]:
import zai
print(zai.__version__)

0.2.2


In [ ]:
map_diagram = {
  "nodes": [
    {
      "id": "N_0",
      "x": 2.2,
      "y": 0.4,
      "span_x": 3.0,
      "span_y": 0.95
    },
    {
      "id": "N_1",
      "x": 2.45,
      "y": 0.4,
      "span_x": 3.0,
      "span_y": 0.7
    },
    {
      "id": "N_2",
      "x": 3.7,
      "y": 0.4,
      "span_x": 3.0,
      "span_y": 0.65
    },
    {
      "id": "N_3",
      "x": 3.95,
      "y": 0.4,
      "span_x": 3.0,
      "span_y": 0.65
    },
    {
      "id": "N_4",
      "x": 7.2,
      "y": 0.4,
      "span_x": 3.0,
      "span_y": 0.65
    },
    {
      "id": "N_5",
      "x": 7.45,
      "y": 0.4,
      "span_x": 3.0,
      "span_y": 0.65
    },
    {
      "id": "N_6",
      "x": 11.4,
      "y": 0.4,
      "span_x": 2.1,
      "span_y": 1.5
    },
    {
      "id": "N_7",
      "x": 11.65,
      "y": 0.4,
      "span_x": 1.85,
      "span_y": 1.5
    },
    {
      "id": "N_8",
      "x": 4.15,
      "y": 0.55,
      "span_x": 3.0,
      "span_y": 0.65
    },
    {
      "id": "N_9",
      "x": 11.9,
      "y": 0.55,
      "span_x": 1.6,
      "span_y": 1.65
    },
    {
      "id": "N_10",
      "x": 11.9,
      "y": 0.8,
      "span_x": 1.6,
      "span_y": 1.9
    },
    {
      "id": "N_11",
      "x": 1.7,
      "y": 0.9,
      "span_x": 2.8,
      "span_y": 2.0
    },
    {
      "id": "N_12",
      "x": 4.5,
      "y": 0.9,
      "span_x": 3.0,
      "span_y": 0.65
    },
    {
      "id": "N_13",
      "x": 6.7,
      "y": 0.9,
      "span_x": 3.0,
      "span_y": 0.65
    },
    {
      "id": "N_14",
      "x": 1.0,
      "y": 1.05,
      "span_x": 2.1,
      "span_y": 2.15
    },
    {
      "id": "N_15",
      "x": 1.4,
      "y": 1.05,
      "span_x": 2.5,
      "span_y": 2.15
    },
    {
      "id": "N_16",
      "x": 4.7,
      "y": 1.05,
      "span_x": 3.0,
      "span_y": 0.65
    },
    {
      "id": "N_17",
      "x": 4.95,
      "y": 1.05,
      "span_x": 3.0,
      "span_y": 0.65
    },
    {
      "id": "N_18",
      "x": 6.25,
      "y": 1.05,
      "span_x": 3.0,
      "span_y": 0.65
    },
    {
      "id": "N_19",
      "x": 6.5,
      "y": 1.05,
      "span_x": 3.0,
      "span_y": 0.65
    },
    {
      "id": "N_20",
      "x": 0.8,
      "y": 1.25,
      "span_x": 1.9,
      "span_y": 2.35
    },
    {
      "id": "N_21",
      "x": 0.6,
      "y": 1.45,
      "span_x": 1.7,
      "span_y": 2.45
    },
    {
      "id": "N_22",
      "x": 0.45,
      "y": 1.65,
      "span_x": 1.55,
      "span_y": 2.5
    },
    {
      "id": "N_23",
      "x": 12.0,
      "y": 1.7,
      "span_x": 1.5,
      "span_y": 2.8
    },
    {
      "id": "N_24",
      "x": 0.4,
      "y": 1.9,
      "span_x": 1.5,
      "span_y": 2.7
    },
    {
      "id": "N_25",
      "x": 11.55,
      "y": 1.9,
      "span_x": 1.95,
      "span_y": 3.0
    },
    {
      "id": "N_26",
      "x": 12.0,
      "y": 3.2,
      "span_x": 1.5,
      "span_y": 3.0
    },
    {
      "id": "N_27",
      "x": 3.9,
      "y": 3.35,
      "span_x": 1.55,
      "span_y": 1.5
    },
    {
      "id": "N_28",
      "x": 4.15,
      "y": 3.35,
      "span_x": 1.8,
      "span_y": 1.5
    },
    {
      "id": "N_29",
      "x": 9.1,
      "y": 3.35,
      "span_x": 3.0,
      "span_y": 1.5
    },
    {
      "id": "N_30",
      "x": 9.35,
      "y": 3.35,
      "span_x": 3.0,
      "span_y": 1.5
    },
    {
      "id": "N_31",
      "x": 9.65,
      "y": 3.35,
      "span_x": 3.0,
      "span_y": 1.5
    },
    {
      "id": "N_32",
      "x": 9.9,
      "y": 3.35,
      "span_x": 3.0,
      "span_y": 1.5
    },
    {
      "id": "N_33",
      "x": 10.15,
      "y": 3.4,
      "span_x": 3.0,
      "span_y": 3.0
    },
    {
      "id": "N_34",
      "x": 0.4,
      "y": 3.45,
      "span_x": 1.5,
      "span_y": 3.0
    },
    {
      "id": "N_35",
      "x": 12.0,
      "y": 3.45,
      "span_x": 1.5,
      "span_y": 3.0
    },
    {
      "id": "N_36",
      "x": 8.9,
      "y": 3.55,
      "span_x": 3.0,
      "span_y": 1.7
    },
    {
      "id": "N_37",
      "x": 1.8,
      "y": 3.6,
      "span_x": 1.95,
      "span_y": 3.0
    },
    {
      "id": "N_38",
      "x": 2.05,
      "y": 3.6,
      "span_x": 1.8,
      "span_y": 1.5
    },
    {
      "id": "N_39",
      "x": 3.85,
      "y": 3.6,
      "span_x": 3.0,
      "span_y": 1.75
    },
    {
      "id": "N_40",
      "x": 5.05,
      "y": 3.6,
      "span_x": 2.05,
      "span_y": 1.75
    },
    {
      "id": "N_41",
      "x": 9.65,
      "y": 3.6,
      "span_x": 3.0,
      "span_y": 1.75
    },
    {
      "id": "N_42",
      "x": 10.35,
      "y": 3.6,
      "span_x": 3.0,
      "span_y": 1.75
    },
    {
      "id": "N_43",
      "x": 11.85,
      "y": 3.65,
      "span_x": 1.7,
      "span_y": 3.0
    },
    {
      "id": "N_44",
      "x": 5.3,
      "y": 3.7,
      "span_x": 2.05,
      "span_y": 1.85
    },
    {
      "id": "N_45",
      "x": 8.6,
      "y": 3.7,
      "span_x": 3.0,
      "span_y": 1.85
    },
    {
      "id": "N_46",
      "x": 0.4,
      "y": 3.75,
      "span_x": 1.5,
      "span_y": 3.0
    },
    {
      "id": "N_47",
      "x": 2.25,
      "y": 3.75,
      "span_x": 1.6,
      "span_y": 1.5
    },
    {
      "id": "N_48",
      "x": 8.85,
      "y": 3.8,
      "span_x": 3.0,
      "span_y": 1.95
    },
    {
      "id": "N_49",
      "x": 3.9,
      "y": 3.85,
      "span_x": 1.55,
      "span_y": 1.95
    },
    {
      "id": "N_50",
      "x": 7.4,
      "y": 3.85,
      "span_x": 3.0,
      "span_y": 1.8
    },
    {
      "id": "N_51",
      "x": 7.65,
      "y": 3.85,
      "span_x": 2.3,
      "span_y": 2.0
    },
    {
      "id": "N_52",
      "x": 8.3,
      "y": 3.85,
      "span_x": 2.95,
      "span_y": 2.0
    },
    {
      "id": "N_53",
      "x": 5.5,
      "y": 3.9,
      "span_x": 3.0,
      "span_y": 2.05
    },
    {
      "id": "N_54",
      "x": 2.35,
      "y": 4.0,
      "span_x": 3.0,
      "span_y": 1.7
    },
    {
      "id": "N_55",
      "x": 9.05,
      "y": 4.0,
      "span_x": 3.0,
      "span_y": 2.1
    },
    {
      "id": "N_56",
      "x": 9.3,
      "y": 4.0,
      "span_x": 3.0,
      "span_y": 2.1
    },
    {
      "id": "N_57",
      "x": 9.85,
      "y": 4.0,
      "span_x": 3.0,
      "span_y": 2.1
    },
    {
      "id": "N_58",
      "x": 10.1,
      "y": 4.0,
      "span_x": 3.0,
      "span_y": 2.1
    },
    {
      "id": "N_59",
      "x": 8.5,
      "y": 4.05,
      "span_x": 3.0,
      "span_y": 2.1
    },
    {
      "id": "N_60",
      "x": 7.35,
      "y": 4.1,
      "span_x": 3.0,
      "span_y": 1.85
    },
    {
      "id": "N_61",
      "x": 9.6,
      "y": 4.15,
      "span_x": 3.0,
      "span_y": 2.1
    },
    {
      "id": "N_62",
      "x": 10.35,
      "y": 4.15,
      "span_x": 3.0,
      "span_y": 2.05
    },
    {
      "id": "N_63",
      "x": 2.35,
      "y": 4.25,
      "span_x": 3.0,
      "span_y": 1.7
    },
    {
      "id": "N_64",
      "x": 11.0,
      "y": 4.25,
      "span_x": 2.8,
      "span_y": 2.9
    },
    {
      "id": "N_65",
      "x": 11.25,
      "y": 4.25,
      "span_x": 2.55,
      "span_y": 2.9
    },
    {
      "id": "N_66",
      "x": 8.6,
      "y": 4.3,
      "span_x": 3.0,
      "span_y": 2.1
    },
    {
      "id": "N_67",
      "x": 7.35,
      "y": 4.35,
      "span_x": 3.0,
      "span_y": 1.85
    },
    {
      "id": "N_68",
      "x": 9.5,
      "y": 4.45,
      "span_x": 3.0,
      "span_y": 2.1
    },
    {
      "id": "N_69",
      "x": 10.65,
      "y": 4.45,
      "span_x": 3.0,
      "span_y": 3.0
    },
    {
      "id": "N_70",
      "x": 2.35,
      "y": 4.55,
      "span_x": 3.0,
      "span_y": 1.7
    },
    {
      "id": "N_71",
      "x": 4.1,
      "y": 4.55,
      "span_x": 1.75,
      "span_y": 2.05
    },
    {
      "id": "N_72",
      "x": 8.6,
      "y": 4.55,
      "span_x": 3.0,
      "span_y": 2.1
    },
    {
      "id": "N_73",
      "x": 7.1,
      "y": 4.6,
      "span_x": 3.0,
      "span_y": 1.75
    },
    {
      "id": "N_74",
      "x": 4.2,
      "y": 4.8,
      "span_x": 1.85,
      "span_y": 2.95
    },
    {
      "id": "N_75",
      "x": 6.9,
      "y": 4.8,
      "span_x": 3.0,
      "span_y": 1.7
    },
    {
      "id": "N_76",
      "x": 9.05,
      "y": 4.8,
      "span_x": 3.0,
      "span_y": 2.1
    },
    {
      "id": "N_77",
      "x": 9.3,
      "y": 4.8,
      "span_x": 3.0,
      "span_y": 2.1
    },
    {
      "id": "N_78",
      "x": 7.65,
      "y": 4.85,
      "span_x": 2.3,
      "span_y": 2.1
    },
    {
      "id": "N_79",
      "x": 10.45,
      "y": 4.85,
      "span_x": 3.0,
      "span_y": 3.0
    },
    {
      "id": "N_80",
      "x": 8.75,
      "y": 4.9,
      "span_x": 3.0,
      "span_y": 2.05
    },
    {
      "id": "N_81",
      "x": 0.4,
      "y": 4.95,
      "span_x": 1.5,
      "span_y": 3.0
    },
    {
      "id": "N_82",
      "x": 4.4,
      "y": 5.0,
      "span_x": 2.1,
      "span_y": 3.0
    },
    {
      "id": "N_83",
      "x": 1.15,
      "y": 5.05,
      "span_x": 1.95,
      "span_y": 3.0
    },
    {
      "id": "N_84",
      "x": 6.85,
      "y": 5.05,
      "span_x": 3.0,
      "span_y": 1.75
    },
    {
      "id": "N_85",
      "x": 7.8,
      "y": 5.05,
      "span_x": 2.45,
      "span_y": 1.9
    },
    {
      "id": "N_86",
      "x": 8.15,
      "y": 5.05,
      "span_x": 2.8,
      "span_y": 1.9
    },
    {
      "id": "N_87",
      "x": 10.25,
      "y": 5.05,
      "span_x": 3.0,
      "span_y": 2.35
    },
    {
      "id": "N_88",
      "x": 12.0,
      "y": 5.05,
      "span_x": 1.8,
      "span_y": 3.0
    },
    {
      "id": "N_89",
      "x": 9.35,
      "y": 5.1,
      "span_x": 3.0,
      "span_y": 1.85
    },
    {
      "id": "N_90",
      "x": 8.45,
      "y": 5.15,
      "span_x": 3.0,
      "span_y": 1.8
    },
    {
      "id": "N_91",
      "x": 9.85,
      "y": 5.15,
      "span_x": 3.0,
      "span_y": 1.8
    },
    {
      "id": "N_92",
      "x": 0.4,
      "y": 5.25,
      "span_x": 1.5,
      "span_y": 3.0
    },
    {
      "id": "N_93",
      "x": 5.95,
      "y": 5.25,
      "span_x": 3.0,
      "span_y": 3.0
    },
    {
      "id": "N_94",
      "x": 6.85,
      "y": 5.3,
      "span_x": 3.0,
      "span_y": 3.0
    },
    {
      "id": "N_95",
      "x": 9.15,
      "y": 5.3,
      "span_x": 3.0,
      "span_y": 3.0
    },
    {
      "id": "N_96",
      "x": 7.1,
      "y": 5.4,
      "span_x": 3.0,
      "span_y": 3.0
    },
    {
      "id": "N_97",
      "x": 5.8,
      "y": 5.45,
      "span_x": 3.0,
      "span_y": 3.0
    },
    {
      "id": "N_98",
      "x": 8.7,
      "y": 5.45,
      "span_x": 3.0,
      "span_y": 3.0
    },
    {
      "id": "N_99",
      "x": 12.3,
      "y": 5.65,
      "span_x": 1.5,
      "span_y": 3.0
    },
    {
      "id": "N_100",
      "x": 11.75,
      "y": 5.85,
      "span_x": 2.05,
      "span_y": 3.0
    },
    {
      "id": "N_101",
      "x": 11.85,
      "y": 6.1,
      "span_x": 1.95,
      "span_y": 3.0
    },
    {
      "id": "N_102",
      "x": 0.4,
      "y": 6.2,
      "span_x": 1.5,
      "span_y": 3.0
    },
    {
      "id": "N_103",
      "x": 3.85,
      "y": 6.3,
      "span_x": 3.0,
      "span_y": 3.0
    },
    {
      "id": "N_104",
      "x": 4.1,
      "y": 6.3,
      "span_x": 1.8,
      "span_y": 3.0
    },
    {
      "id": "N_105",
      "x": 11.9,
      "y": 6.35,
      "span_x": 1.9,
      "span_y": 3.0
    },
    {
      "id": "N_106",
      "x": 0.4,
      "y": 6.45,
      "span_x": 1.5,
      "span_y": 3.0
    },
    {
      "id": "N_107",
      "x": 4.35,
      "y": 6.45,
      "span_x": 2.05,
      "span_y": 3.0
    },
    {
      "id": "N_108",
      "x": 3.8,
      "y": 6.55,
      "span_x": 3.0,
      "span_y": 3.0
    },
    {
      "id": "N_109",
      "x": 4.7,
      "y": 6.6,
      "span_x": 2.4,
      "span_y": 3.0
    },
    {
      "id": "N_110",
      "x": 11.7,
      "y": 6.65,
      "span_x": 2.1,
      "span_y": 3.0
    },
    {
      "id": "N_111",
      "x": 0.6,
      "y": 6.7,
      "span_x": 1.7,
      "span_y": 3.0
    },
    {
      "id": "N_112",
      "x": 1.8,
      "y": 6.7,
      "span_x": 2.0,
      "span_y": 3.0
    },
    {
      "id": "N_113",
      "x": 2.05,
      "y": 6.7,
      "span_x": 1.85,
      "span_y": 1.25
    },
    {
      "id": "N_114",
      "x": 4.35,
      "y": 6.7,
      "span_x": 2.05,
      "span_y": 3.0
    },
    {
      "id": "N_115",
      "x": 4.95,
      "y": 6.7,
      "span_x": 2.65,
      "span_y": 3.0
    },
    {
      "id": "N_116",
      "x": 5.2,
      "y": 6.7,
      "span_x": 2.9,
      "span_y": 3.0
    },
    {
      "id": "N_117",
      "x": 5.45,
      "y": 6.7,
      "span_x": 3.0,
      "span_y": 3.0
    },
    {
      "id": "N_118",
      "x": 7.25,
      "y": 6.7,
      "span_x": 3.0,
      "span_y": 3.0
    },
    {
      "id": "N_119",
      "x": 7.5,
      "y": 6.7,
      "span_x": 3.0,
      "span_y": 3.0
    },
    {
      "id": "N_120",
      "x": 8.9,
      "y": 6.7,
      "span_x": 3.0,
      "span_y": 2.15
    },
    {
      "id": "N_121",
      "x": 9.15,
      "y": 6.7,
      "span_x": 3.0,
      "span_y": 2.15
    },
    {
      "id": "N_122",
      "x": 9.95,
      "y": 6.75,
      "span_x": 3.0,
      "span_y": 2.1
    },
    {
      "id": "N_123",
      "x": 1.3,
      "y": 6.8,
      "span_x": 2.0,
      "span_y": 3.0
    },
    {
      "id": "N_124",
      "x": 2.3,
      "y": 6.8,
      "span_x": 3.0,
      "span_y": 1.55
    },
    {
      "id": "N_125",
      "x": 10.2,
      "y": 6.8,
      "span_x": 3.0,
      "span_y": 2.05
    },
    {
      "id": "N_126",
      "x": 1.55,
      "y": 6.85,
      "span_x": 2.0,
      "span_y": 3.0
    },
    {
      "id": "N_127",
      "x": 3.95,
      "y": 6.85,
      "span_x": 1.65,
      "span_y": 3.0
    },
    {
      "id": "N_128",
      "x": 9.35,
      "y": 6.85,
      "span_x": 3.0,
      "span_y": 2.0
    },
    {
      "id": "N_129",
      "x": 11.9,
      "y": 6.85,
      "span_x": 1.9,
      "span_y": 3.0
    },
    {
      "id": "N_130",
      "x": 0.45,
      "y": 6.9,
      "span_x": 1.55,
      "span_y": 3.0
    },
    {
      "id": "N_131",
      "x": 4.8,
      "y": 6.9,
      "span_x": 2.5,
      "span_y": 3.0
    },
    {
      "id": "N_132",
      "x": 7.2,
      "y": 6.95,
      "span_x": 3.0,
      "span_y": 3.0
    },
    {
      "id": "N_133",
      "x": 9.75,
      "y": 6.95,
      "span_x": 3.0,
      "span_y": 1.9
    },
    {
      "id": "N_134",
      "x": 2.4,
      "y": 7.05,
      "span_x": 3.0,
      "span_y": 1.6
    },
    {
      "id": "N_135",
      "x": 11.9,
      "y": 7.1,
      "span_x": 1.9,
      "span_y": 3.0
    },
    {
      "id": "N_136",
      "x": 0.4,
      "y": 7.15,
      "span_x": 1.5,
      "span_y": 3.0
    },
    {
      "id": "N_137",
      "x": 4.8,
      "y": 7.15,
      "span_x": 2.5,
      "span_y": 2.35
    },
    {
      "id": "N_138",
      "x": 6.45,
      "y": 7.2,
      "span_x": 3.0,
      "span_y": 2.0
    },
    {
      "id": "N_139",
      "x": 6.7,
      "y": 7.2,
      "span_x": 3.0,
      "span_y": 2.0
    },
    {
      "id": "N_140",
      "x": 7.15,
      "y": 7.2,
      "span_x": 3.0,
      "span_y": 2.0
    },
    {
      "id": "N_141",
      "x": 9.6,
      "y": 7.25,
      "span_x": 3.0,
      "span_y": 0.65
    },
    {
      "id": "N_142",
      "x": 3.95,
      "y": 7.3,
      "span_x": 1.65,
      "span_y": 2.5
    },
    {
      "id": "N_143",
      "x": 4.9,
      "y": 7.4,
      "span_x": 2.6,
      "span_y": 2.6
    },
    {
      "id": "N_144",
      "x": 3.95,
      "y": 7.55,
      "span_x": 1.65,
      "span_y": 2.75
    },
    {
      "id": "N_145",
      "x": 4.85,
      "y": 7.7,
      "span_x": 2.55,
      "span_y": 2.9
    },
    {
      "id": "N_146",
      "x": 2.4,
      "y": 7.9,
      "span_x": 3.0,
      "span_y": 3.0
    },
    {
      "id": "N_147",
      "x": 0.4,
      "y": 8.7,
      "span_x": 1.5,
      "span_y": 2.05
    },
    {
      "id": "N_148",
      "x": 4.1,
      "y": 8.85,
      "span_x": 3.0,
      "span_y": 2.25
    },
    {
      "id": "N_149",
      "x": 6.4,
      "y": 8.85,
      "span_x": 3.0,
      "span_y": 2.3
    },
    {
      "id": "N_150",
      "x": 0.4,
      "y": 8.95,
      "span_x": 1.5,
      "span_y": 1.8
    },
    {
      "id": "N_151",
      "x": 7.2,
      "y": 8.95,
      "span_x": 3.0,
      "span_y": 2.2
    },
    {
      "id": "N_152",
      "x": 11.9,
      "y": 9.0,
      "span_x": 1.5,
      "span_y": 2.0
    },
    {
      "id": "N_153",
      "x": 4.1,
      "y": 9.1,
      "span_x": 3.0,
      "span_y": 2.0
    },
    {
      "id": "N_154",
      "x": 6.2,
      "y": 9.1,
      "span_x": 3.0,
      "span_y": 2.05
    },
    {
      "id": "N_155",
      "x": 0.55,
      "y": 9.15,
      "span_x": 1.65,
      "span_y": 1.75
    },
    {
      "id": "N_156",
      "x": 0.8,
      "y": 9.25,
      "span_x": 1.9,
      "span_y": 1.75
    },
    {
      "id": "N_157",
      "x": 1.6,
      "y": 9.25,
      "span_x": 2.7,
      "span_y": 1.75
    },
    {
      "id": "N_158",
      "x": 1.85,
      "y": 9.25,
      "span_x": 2.95,
      "span_y": 1.75
    },
    {
      "id": "N_159",
      "x": 4.35,
      "y": 9.25,
      "span_x": 3.0,
      "span_y": 1.85
    },
    {
      "id": "N_160",
      "x": 11.9,
      "y": 9.25,
      "span_x": 1.5,
      "span_y": 1.75
    },
    {
      "id": "N_161",
      "x": 7.25,
      "y": 9.3,
      "span_x": 3.0,
      "span_y": 1.85
    },
    {
      "id": "N_162",
      "x": 4.1,
      "y": 9.4,
      "span_x": 3.0,
      "span_y": 1.7
    },
    {
      "id": "N_163",
      "x": 4.6,
      "y": 9.45,
      "span_x": 3.0,
      "span_y": 1.65
    },
    {
      "id": "N_164",
      "x": 2.15,
      "y": 9.5,
      "span_x": 3.0,
      "span_y": 0.25
    },
    {
      "id": "N_165",
      "x": 2.4,
      "y": 9.5,
      "span_x": 3.0,
      "span_y": 0.25
    },
    {
      "id": "N_166",
      "x": 3.7,
      "y": 9.5,
      "span_x": 3.0,
      "span_y": 1.5
    },
    {
      "id": "N_167",
      "x": 4.85,
      "y": 9.5,
      "span_x": 3.0,
      "span_y": 1.6
    },
    {
      "id": "N_168",
      "x": 7.05,
      "y": 9.5,
      "span_x": 3.0,
      "span_y": 1.65
    },
    {
      "id": "N_169",
      "x": 7.4,
      "y": 9.5,
      "span_x": 3.0,
      "span_y": 1.65
    },
    {
      "id": "N_170",
      "x": 7.65,
      "y": 9.5,
      "span_x": 3.0,
      "span_y": 1.65
    },
    {
      "id": "N_171",
      "x": 11.4,
      "y": 9.5,
      "span_x": 2.0,
      "span_y": 1.5
    },
    {
      "id": "N_172",
      "x": 11.65,
      "y": 9.5,
      "span_x": 1.75,
      "span_y": 1.5
    },
    {
      "id": "N_173",
      "x": 6.25,
      "y": 9.55,
      "span_x": 3.0,
      "span_y": 1.6
    },
    {
      "id": "N_174",
      "x": 6.5,
      "y": 9.65,
      "span_x": 3.0,
      "span_y": 1.5
    },
    {
      "id": "N_175",
      "x": 6.75,
      "y": 9.65,
      "span_x": 3.0,
      "span_y": 1.5
    }
  ],
  "edges": [
    {
      "from": "N_38",
      "to": "N_38"
    },
    {
      "from": "N_69",
      "to": "N_69"
    },
    {
      "from": "N_7",
      "to": "N_7"
    },
    {
      "from": "N_55",
      "to": "N_56"
    },
    {
      "from": "N_34",
      "to": "N_34"
    },
    {
      "from": "N_33",
      "to": "N_42"
    },
    {
      "from": "N_0",
      "to": "N_1"
    },
    {
      "from": "N_132",
      "to": "N_132"
    },
    {
      "from": "N_156",
      "to": "N_157"
    },
    {
      "from": "N_59",
      "to": "N_66"
    },
    {
      "from": "N_158",
      "to": "N_164"
    },
    {
      "from": "N_113",
      "to": "N_124"
    },
    {
      "from": "N_61",
      "to": "N_61"
    },
    {
      "from": "N_169",
      "to": "N_169"
    },
    {
      "from": "N_121",
      "to": "N_128"
    },
    {
      "from": "N_103",
      "to": "N_108"
    },
    {
      "from": "N_32",
      "to": "N_33"
    },
    {
      "from": "N_122",
      "to": "N_122"
    },
    {
      "from": "N_57",
      "to": "N_61"
    },
    {
      "from": "N_51",
      "to": "N_52"
    },
    {
      "from": "N_166",
      "to": "N_166"
    },
    {
      "from": "N_71",
      "to": "N_71"
    },
    {
      "from": "N_55",
      "to": "N_55"
    },
    {
      "from": "N_35",
      "to": "N_35"
    },
    {
      "from": "N_0",
      "to": "N_0"
    },
    {
      "from": "N_160",
      "to": "N_160"
    },
    {
      "from": "N_16",
      "to": "N_17"
    },
    {
      "from": "N_75",
      "to": "N_84"
    },
    {
      "from": "N_127",
      "to": "N_142"
    },
    {
      "from": "N_69",
      "to": "N_79"
    },
    {
      "from": "N_165",
      "to": "N_165"
    },
    {
      "from": "N_21",
      "to": "N_22"
    },
    {
      "from": "N_162",
      "to": "N_162"
    },
    {
      "from": "N_153",
      "to": "N_159"
    },
    {
      "from": "N_64",
      "to": "N_64"
    },
    {
      "from": "N_100",
      "to": "N_100"
    },
    {
      "from": "N_118",
      "to": "N_119"
    },
    {
      "from": "N_20",
      "to": "N_21"
    },
    {
      "from": "N_152",
      "to": "N_152"
    },
    {
      "from": "N_120",
      "to": "N_121"
    },
    {
      "from": "N_16",
      "to": "N_16"
    },
    {
      "from": "N_14",
      "to": "N_20"
    },
    {
      "from": "N_22",
      "to": "N_22"
    },
    {
      "from": "N_30",
      "to": "N_30"
    },
    {
      "from": "N_118",
      "to": "N_132"
    },
    {
      "from": "N_167",
      "to": "N_173"
    },
    {
      "from": "N_11",
      "to": "N_11"
    },
    {
      "from": "N_80",
      "to": "N_80"
    },
    {
      "from": "N_110",
      "to": "N_125"
    },
    {
      "from": "N_139",
      "to": "N_139"
    },
    {
      "from": "N_118",
      "to": "N_118"
    },
    {
      "from": "N_12",
      "to": "N_12"
    },
    {
      "from": "N_120",
      "to": "N_120"
    },
    {
      "from": "N_9",
      "to": "N_9"
    },
    {
      "from": "N_95",
      "to": "N_95"
    },
    {
      "from": "N_37",
      "to": "N_38"
    },
    {
      "from": "N_48",
      "to": "N_48"
    },
    {
      "from": "N_134",
      "to": "N_134"
    },
    {
      "from": "N_27",
      "to": "N_39"
    },
    {
      "from": "N_144",
      "to": "N_144"
    },
    {
      "from": "N_57",
      "to": "N_58"
    },
    {
      "from": "N_159",
      "to": "N_162"
    },
    {
      "from": "N_101",
      "to": "N_105"
    },
    {
      "from": "N_8",
      "to": "N_12"
    },
    {
      "from": "N_136",
      "to": "N_147"
    },
    {
      "from": "N_102",
      "to": "N_106"
    },
    {
      "from": "N_66",
      "to": "N_66"
    },
    {
      "from": "N_60",
      "to": "N_60"
    },
    {
      "from": "N_37",
      "to": "N_37"
    },
    {
      "from": "N_137",
      "to": "N_143"
    },
    {
      "from": "N_154",
      "to": "N_154"
    },
    {
      "from": "N_50",
      "to": "N_51"
    },
    {
      "from": "N_57",
      "to": "N_57"
    },
    {
      "from": "N_114",
      "to": "N_114"
    },
    {
      "from": "N_80",
      "to": "N_90"
    },
    {
      "from": "N_49",
      "to": "N_71"
    },
    {
      "from": "N_88",
      "to": "N_99"
    },
    {
      "from": "N_164",
      "to": "N_165"
    },
    {
      "from": "N_59",
      "to": "N_59"
    },
    {
      "from": "N_46",
      "to": "N_81"
    },
    {
      "from": "N_81",
      "to": "N_83"
    },
    {
      "from": "N_124",
      "to": "N_134"
    },
    {
      "from": "N_54",
      "to": "N_54"
    },
    {
      "from": "N_50",
      "to": "N_50"
    },
    {
      "from": "N_129",
      "to": "N_135"
    },
    {
      "from": "N_151",
      "to": "N_151"
    },
    {
      "from": "N_29",
      "to": "N_30"
    },
    {
      "from": "N_14",
      "to": "N_14"
    },
    {
      "from": "N_159",
      "to": "N_159"
    },
    {
      "from": "N_164",
      "to": "N_164"
    },
    {
      "from": "N_75",
      "to": "N_75"
    },
    {
      "from": "N_109",
      "to": "N_109"
    },
    {
      "from": "N_91",
      "to": "N_87"
    },
    {
      "from": "N_81",
      "to": "N_81"
    },
    {
      "from": "N_122",
      "to": "N_133"
    },
    {
      "from": "N_35",
      "to": "N_43"
    },
    {
      "from": "N_142",
      "to": "N_145"
    },
    {
      "from": "N_87",
      "to": "N_87"
    },
    {
      "from": "N_3",
      "to": "N_3"
    },
    {
      "from": "N_77",
      "to": "N_77"
    },
    {
      "from": "N_163",
      "to": "N_167"
    },
    {
      "from": "N_29",
      "to": "N_29"
    },
    {
      "from": "N_39",
      "to": "N_39"
    },
    {
      "from": "N_1",
      "to": "N_2"
    },
    {
      "from": "N_98",
      "to": "N_98"
    },
    {
      "from": "N_116",
      "to": "N_116"
    },
    {
      "from": "N_10",
      "to": "N_10"
    },
    {
      "from": "N_3",
      "to": "N_8"
    },
    {
      "from": "N_106",
      "to": "N_106"
    },
    {
      "from": "N_133",
      "to": "N_133"
    },
    {
      "from": "N_73",
      "to": "N_75"
    },
    {
      "from": "N_42",
      "to": "N_64"
    },
    {
      "from": "N_138",
      "to": "N_139"
    },
    {
      "from": "N_81",
      "to": "N_92"
    },
    {
      "from": "N_17",
      "to": "N_18"
    },
    {
      "from": "N_130",
      "to": "N_130"
    },
    {
      "from": "N_166",
      "to": "N_162"
    },
    {
      "from": "N_85",
      "to": "N_86"
    },
    {
      "from": "N_101",
      "to": "N_101"
    },
    {
      "from": "N_79",
      "to": "N_87"
    },
    {
      "from": "N_34",
      "to": "N_37"
    },
    {
      "from": "N_68",
      "to": "N_77"
    },
    {
      "from": "N_138",
      "to": "N_138"
    },
    {
      "from": "N_67",
      "to": "N_67"
    },
    {
      "from": "N_44",
      "to": "N_53"
    },
    {
      "from": "N_105",
      "to": "N_110"
    },
    {
      "from": "N_71",
      "to": "N_74"
    },
    {
      "from": "N_5",
      "to": "N_5"
    },
    {
      "from": "N_79",
      "to": "N_79"
    },
    {
      "from": "N_65",
      "to": "N_65"
    },
    {
      "from": "N_64",
      "to": "N_69"
    },
    {
      "from": "N_104",
      "to": "N_107"
    },
    {
      "from": "N_140",
      "to": "N_140"
    },
    {
      "from": "N_123",
      "to": "N_126"
    },
    {
      "from": "N_89",
      "to": "N_95"
    },
    {
      "from": "N_175",
      "to": "N_175"
    },
    {
      "from": "N_2",
      "to": "N_3"
    },
    {
      "from": "N_76",
      "to": "N_77"
    },
    {
      "from": "N_150",
      "to": "N_150"
    },
    {
      "from": "N_163",
      "to": "N_163"
    },
    {
      "from": "N_78",
      "to": "N_85"
    },
    {
      "from": "N_115",
      "to": "N_116"
    },
    {
      "from": "N_173",
      "to": "N_173"
    },
    {
      "from": "N_139",
      "to": "N_140"
    },
    {
      "from": "N_160",
      "to": "N_172"
    },
    {
      "from": "N_124",
      "to": "N_124"
    },
    {
      "from": "N_50",
      "to": "N_60"
    },
    {
      "from": "N_28",
      "to": "N_40"
    },
    {
      "from": "N_104",
      "to": "N_104"
    },
    {
      "from": "N_13",
      "to": "N_13"
    },
    {
      "from": "N_122",
      "to": "N_125"
    },
    {
      "from": "N_84",
      "to": "N_84"
    },
    {
      "from": "N_113",
      "to": "N_113"
    },
    {
      "from": "N_151",
      "to": "N_161"
    },
    {
      "from": "N_2",
      "to": "N_2"
    },
    {
      "from": "N_76",
      "to": "N_76"
    },
    {
      "from": "N_47",
      "to": "N_54"
    },
    {
      "from": "N_18",
      "to": "N_19"
    },
    {
      "from": "N_165",
      "to": "N_166"
    },
    {
      "from": "N_107",
      "to": "N_114"
    },
    {
      "from": "N_115",
      "to": "N_115"
    },
    {
      "from": "N_31",
      "to": "N_41"
    },
    {
      "from": "N_11",
      "to": "N_15"
    },
    {
      "from": "N_105",
      "to": "N_105"
    },
    {
      "from": "N_108",
      "to": "N_127"
    },
    {
      "from": "N_65",
      "to": "N_64"
    },
    {
      "from": "N_54",
      "to": "N_63"
    },
    {
      "from": "N_110",
      "to": "N_110"
    },
    {
      "from": "N_135",
      "to": "N_135"
    },
    {
      "from": "N_112",
      "to": "N_126"
    },
    {
      "from": "N_22",
      "to": "N_24"
    },
    {
      "from": "N_111",
      "to": "N_111"
    },
    {
      "from": "N_136",
      "to": "N_136"
    },
    {
      "from": "N_44",
      "to": "N_44"
    },
    {
      "from": "N_117",
      "to": "N_138"
    },
    {
      "from": "N_60",
      "to": "N_67"
    },
    {
      "from": "N_36",
      "to": "N_36"
    },
    {
      "from": "N_5",
      "to": "N_6"
    },
    {
      "from": "N_147",
      "to": "N_147"
    },
    {
      "from": "N_23",
      "to": "N_25"
    },
    {
      "from": "N_4",
      "to": "N_5"
    },
    {
      "from": "N_89",
      "to": "N_89"
    },
    {
      "from": "N_26",
      "to": "N_35"
    },
    {
      "from": "N_142",
      "to": "N_142"
    },
    {
      "from": "N_156",
      "to": "N_156"
    },
    {
      "from": "N_6",
      "to": "N_7"
    },
    {
      "from": "N_115",
      "to": "N_131"
    },
    {
      "from": "N_88",
      "to": "N_88"
    },
    {
      "from": "N_52",
      "to": "N_59"
    },
    {
      "from": "N_133",
      "to": "N_141"
    },
    {
      "from": "N_170",
      "to": "N_171"
    },
    {
      "from": "N_174",
      "to": "N_175"
    },
    {
      "from": "N_111",
      "to": "N_123"
    },
    {
      "from": "N_45",
      "to": "N_52"
    },
    {
      "from": "N_130",
      "to": "N_136"
    },
    {
      "from": "N_159",
      "to": "N_163"
    },
    {
      "from": "N_89",
      "to": "N_91"
    },
    {
      "from": "N_167",
      "to": "N_167"
    },
    {
      "from": "N_135",
      "to": "N_152"
    },
    {
      "from": "N_39",
      "to": "N_49"
    },
    {
      "from": "N_6",
      "to": "N_6"
    },
    {
      "from": "N_78",
      "to": "N_78"
    },
    {
      "from": "N_107",
      "to": "N_107"
    },
    {
      "from": "N_103",
      "to": "N_104"
    },
    {
      "from": "N_112",
      "to": "N_113"
    },
    {
      "from": "N_72",
      "to": "N_72"
    },
    {
      "from": "N_174",
      "to": "N_174"
    },
    {
      "from": "N_15",
      "to": "N_15"
    },
    {
      "from": "N_161",
      "to": "N_169"
    },
    {
      "from": "N_108",
      "to": "N_108"
    },
    {
      "from": "N_172",
      "to": "N_172"
    },
    {
      "from": "N_63",
      "to": "N_70"
    },
    {
      "from": "N_91",
      "to": "N_91"
    },
    {
      "from": "N_128",
      "to": "N_133"
    },
    {
      "from": "N_103",
      "to": "N_103"
    },
    {
      "from": "N_83",
      "to": "N_83"
    },
    {
      "from": "N_43",
      "to": "N_65"
    },
    {
      "from": "N_112",
      "to": "N_112"
    },
    {
      "from": "N_148",
      "to": "N_148"
    },
    {
      "from": "N_23",
      "to": "N_23"
    },
    {
      "from": "N_14",
      "to": "N_15"
    },
    {
      "from": "N_67",
      "to": "N_73"
    },
    {
      "from": "N_68",
      "to": "N_69"
    },
    {
      "from": "N_134",
      "to": "N_146"
    },
    {
      "from": "N_109",
      "to": "N_115"
    },
    {
      "from": "N_13",
      "to": "N_19"
    },
    {
      "from": "N_114",
      "to": "N_127"
    },
    {
      "from": "N_34",
      "to": "N_46"
    },
    {
      "from": "N_95",
      "to": "N_98"
    },
    {
      "from": "N_32",
      "to": "N_32"
    },
    {
      "from": "N_100",
      "to": "N_101"
    },
    {
      "from": "N_20",
      "to": "N_20"
    },
    {
      "from": "N_62",
      "to": "N_69"
    },
    {
      "from": "N_43",
      "to": "N_43"
    },
    {
      "from": "N_36",
      "to": "N_48"
    },
    {
      "from": "N_161",
      "to": "N_168"
    },
    {
      "from": "N_24",
      "to": "N_24"
    },
    {
      "from": "N_158",
      "to": "N_158"
    },
    {
      "from": "N_92",
      "to": "N_102"
    },
    {
      "from": "N_168",
      "to": "N_168"
    },
    {
      "from": "N_84",
      "to": "N_94"
    },
    {
      "from": "N_45",
      "to": "N_45"
    },
    {
      "from": "N_65",
      "to": "N_88"
    },
    {
      "from": "N_85",
      "to": "N_85"
    },
    {
      "from": "N_141",
      "to": "N_141"
    },
    {
      "from": "N_140",
      "to": "N_151"
    },
    {
      "from": "N_155",
      "to": "N_155"
    },
    {
      "from": "N_86",
      "to": "N_86"
    },
    {
      "from": "N_68",
      "to": "N_68"
    },
    {
      "from": "N_119",
      "to": "N_120"
    },
    {
      "from": "N_150",
      "to": "N_155"
    },
    {
      "from": "N_66",
      "to": "N_72"
    },
    {
      "from": "N_175",
      "to": "N_168"
    },
    {
      "from": "N_171",
      "to": "N_172"
    },
    {
      "from": "N_58",
      "to": "N_62"
    },
    {
      "from": "N_96",
      "to": "N_96"
    },
    {
      "from": "N_155",
      "to": "N_156"
    },
    {
      "from": "N_70",
      "to": "N_70"
    },
    {
      "from": "N_144",
      "to": "N_148"
    },
    {
      "from": "N_128",
      "to": "N_128"
    },
    {
      "from": "N_82",
      "to": "N_82"
    },
    {
      "from": "N_132",
      "to": "N_140"
    },
    {
      "from": "N_131",
      "to": "N_137"
    },
    {
      "from": "N_28",
      "to": "N_28"
    },
    {
      "from": "N_171",
      "to": "N_171"
    },
    {
      "from": "N_61",
      "to": "N_68"
    },
    {
      "from": "N_0",
      "to": "N_11"
    },
    {
      "from": "N_33",
      "to": "N_33"
    },
    {
      "from": "N_1",
      "to": "N_1"
    },
    {
      "from": "N_102",
      "to": "N_102"
    },
    {
      "from": "N_94",
      "to": "N_96"
    },
    {
      "from": "N_119",
      "to": "N_119"
    },
    {
      "from": "N_47",
      "to": "N_47"
    },
    {
      "from": "N_31",
      "to": "N_32"
    },
    {
      "from": "N_18",
      "to": "N_18"
    },
    {
      "from": "N_40",
      "to": "N_40"
    },
    {
      "from": "N_106",
      "to": "N_111"
    },
    {
      "from": "N_40",
      "to": "N_44"
    },
    {
      "from": "N_157",
      "to": "N_158"
    },
    {
      "from": "N_74",
      "to": "N_82"
    },
    {
      "from": "N_82",
      "to": "N_97"
    },
    {
      "from": "N_23",
      "to": "N_26"
    },
    {
      "from": "N_131",
      "to": "N_131"
    },
    {
      "from": "N_52",
      "to": "N_52"
    },
    {
      "from": "N_142",
      "to": "N_144"
    },
    {
      "from": "N_31",
      "to": "N_31"
    },
    {
      "from": "N_153",
      "to": "N_153"
    },
    {
      "from": "N_154",
      "to": "N_173"
    },
    {
      "from": "N_161",
      "to": "N_161"
    },
    {
      "from": "N_4",
      "to": "N_4"
    },
    {
      "from": "N_24",
      "to": "N_34"
    },
    {
      "from": "N_48",
      "to": "N_55"
    },
    {
      "from": "N_157",
      "to": "N_157"
    },
    {
      "from": "N_26",
      "to": "N_26"
    },
    {
      "from": "N_173",
      "to": "N_174"
    },
    {
      "from": "N_94",
      "to": "N_94"
    },
    {
      "from": "N_53",
      "to": "N_53"
    },
    {
      "from": "N_77",
      "to": "N_89"
    },
    {
      "from": "N_121",
      "to": "N_121"
    },
    {
      "from": "N_4",
      "to": "N_13"
    },
    {
      "from": "N_73",
      "to": "N_73"
    },
    {
      "from": "N_152",
      "to": "N_160"
    },
    {
      "from": "N_45",
      "to": "N_36"
    },
    {
      "from": "N_74",
      "to": "N_74"
    },
    {
      "from": "N_62",
      "to": "N_62"
    },
    {
      "from": "N_147",
      "to": "N_150"
    },
    {
      "from": "N_110",
      "to": "N_129"
    },
    {
      "from": "N_12",
      "to": "N_16"
    },
    {
      "from": "N_29",
      "to": "N_36"
    },
    {
      "from": "N_111",
      "to": "N_130"
    },
    {
      "from": "N_27",
      "to": "N_28"
    },
    {
      "from": "N_116",
      "to": "N_117"
    },
    {
      "from": "N_67",
      "to": "N_78"
    },
    {
      "from": "N_76",
      "to": "N_80"
    },
    {
      "from": "N_53",
      "to": "N_93"
    },
    {
      "from": "N_148",
      "to": "N_153"
    },
    {
      "from": "N_58",
      "to": "N_58"
    },
    {
      "from": "N_127",
      "to": "N_127"
    },
    {
      "from": "N_123",
      "to": "N_123"
    },
    {
      "from": "N_27",
      "to": "N_27"
    },
    {
      "from": "N_7",
      "to": "N_9"
    },
    {
      "from": "N_38",
      "to": "N_47"
    },
    {
      "from": "N_149",
      "to": "N_151"
    },
    {
      "from": "N_30",
      "to": "N_31"
    },
    {
      "from": "N_107",
      "to": "N_109"
    },
    {
      "from": "N_9",
      "to": "N_10"
    },
    {
      "from": "N_21",
      "to": "N_21"
    },
    {
      "from": "N_137",
      "to": "N_137"
    },
    {
      "from": "N_169",
      "to": "N_170"
    },
    {
      "from": "N_10",
      "to": "N_23"
    },
    {
      "from": "N_129",
      "to": "N_129"
    },
    {
      "from": "N_64",
      "to": "N_65"
    },
    {
      "from": "N_51",
      "to": "N_51"
    }
  ]
}

In [ ]:
from zai import ZhipuAiClient
import json

# 初始化客户端
client = ZhipuAiClient(api_key="d2ce696b586b4657a6ec77005f26dd82.zPMQ531oxnTYfs5v")

# 基础 JSON 模式
response = client.chat.completions.create(
    model="GLM-4-Flash-250414",
    messages=[
        {
            "role": "system",
            "content": f"""
            你是一个情感分析专家。请根据以下这张结构化图，给出：{map_diagram}
            """
        },
        {
            "role": "user",
            "content": "请分析这句话的情感：'我就一扫大街的'"
        }
    ],
    # response_format={
    #     "type": "json_object"
    # }
)

# 解析结果
result = json.loads(response.choices[0].message.content)
print(f"情感: {result['sentiment']}")
print(f"置信度: {result['confidence']}")
print(f"情绪: {result['emotions']}")

情感: neutral
置信度: 0.85
情绪: ['indifference']
